# 05 — Production Patterns

**What you'll learn**
- Five patterns that separate a hobby MCP server from one you'd put behind an LLM that touches real money.
- Input validation with pydantic — never trust what the model emits.
- Structured logging so you can answer "what did the agent do last Tuesday?".
- Typed error handling so failures don't leak stack traces.
- Tool versioning so you can evolve your API without breaking callers.
- Read vs write separation so a curious agent can browse without nuking your CRM.

## Pattern 1 — Input validation with pydantic

LLMs hallucinate. Tool arguments are the model's free-form output, validated by you. Pydantic gives you typed models with friendly error messages.

**Prereq:** this notebook needs pydantic. If the import below fails, run this in a notebook cell first:

```python
%pip install --quiet pydantic "pydantic[email]"
```


In [ ]:
from pydantic import BaseModel, EmailStr, Field, ValidationError


class CreateContactArgs(BaseModel):
    name: str = Field(min_length=1, max_length=200)
    email: EmailStr


class AssignOSCArgs(BaseModel):
    contact_id: str = Field(pattern=r"^contact_\d+$")


class CreateFollowupArgs(BaseModel):
    contact_id: str = Field(pattern=r"^contact_\d+$")
    note: str = Field(min_length=1, max_length=1000)


# Try a valid one and an invalid one to feel the difference.
print(CreateContactArgs(name="John", email="john@example.com"))

try:
    CreateContactArgs(name="", email="not-an-email")
except ValidationError as e:
    print(e)

## Pattern 2 — Structured logging

Every tool call should leave a trail: who called, which tool, with what arguments, how long it took, and whether it succeeded. Plain `print` is fine for a demo; use `logging` or a structured logger (`structlog`, `loguru`) in production.

In [ ]:
import logging, time, json

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("mcp")


def logged(tool_name: str):
    def deco(fn):
        def wrapper(**kwargs):
            t0 = time.perf_counter()
            status = "ok"
            try:
                return fn(**kwargs)
            except Exception:
                status = "error"
                raise
            finally:
                dur_ms = int((time.perf_counter() - t0) * 1000)
                log.info(json.dumps({
                    "tool": tool_name,
                    "args": kwargs,
                    "status": status,
                    "duration_ms": dur_ms,
                }))
        return wrapper
    return deco

## Pattern 3 — Typed errors, no stack traces

If a contact doesn't exist, the client should see a clean error code, not a Python traceback. Wrap exceptions at the MCP boundary and translate them.

In [ ]:
class ToolError(Exception):
    """Translatable to an MCP error response."""

    def __init__(self, code: str, message: str, *, retryable: bool = False):
        super().__init__(message)
        self.code = code
        self.retryable = retryable

    def to_response(self) -> dict:
        return {"error": {"code": self.code, "message": str(self),
                          "retryable": self.retryable}}


class NotFoundError(ToolError):
    def __init__(self, what: str):
        super().__init__("not_found", what, retryable=False)


class ValidationFailed(ToolError):
    def __init__(self, message: str):
        super().__init__("validation_failed", message, retryable=False)


def safe_call(fn, **kwargs):
    """Run a tool; convert any error to a clean structured response."""
    try:
        return {"ok": True, "result": fn(**kwargs)}
    except ValidationError as e:
        return ValidationFailed(str(e)).to_response()
    except ToolError as e:
        return e.to_response()
    except Exception as e:
        # Unexpected — log internally, return generic message externally.
        log.exception("unexpected error in tool")
        return ToolError("internal_error", "internal error", retryable=True).to_response()

## Pattern 4 — Tool versioning

Once a real LLM agent depends on your tool, **you cannot change its signature in place**. Add a new version and deprecate the old one over time.

Here we evolve `create_contact` to require a phone number — but only in v2. v1 stays available for callers we haven't migrated.

In [ ]:
# Fake in-memory CRM. In production this would be HubSpot, Salesforce, etc.
CONTACTS: dict = {}
TASKS: list = []
OSC_TEAM = [
    {"id": "osc_101", "name": "Ava OSC",  "last_assigned": 0},
    {"id": "osc_102", "name": "Ben OSC",  "last_assigned": 0},
    {"id": "osc_103", "name": "Cara OSC", "last_assigned": 0},
]
_assignment_counter = 0
print("Fake CRM ready. Contacts:", len(CONTACTS), "OSCs:", len(OSC_TEAM))

In [ ]:
@logged("create_contact_v1")
def create_contact_v1(name: str, email: str) -> dict:
    """v1 — name and email only."""
    args = CreateContactArgs(name=name, email=email)
    cid = f"contact_{len(CONTACTS) + 1}"
    contact = {"id": cid, "name": args.name, "email": args.email,
               "phone": None, "owner_id": None, "version": "v1"}
    CONTACTS[cid] = contact
    return contact


class CreateContactArgsV2(BaseModel):
    name: str = Field(min_length=1)
    email: EmailStr
    phone: str = Field(pattern=r"^\+?[0-9\-\s]{7,}$")


@logged("create_contact_v2")
def create_contact_v2(name: str, email: str, phone: str) -> dict:
    """v2 — phone is required."""
    args = CreateContactArgsV2(name=name, email=email, phone=phone)
    cid = f"contact_{len(CONTACTS) + 1}"
    contact = {"id": cid, "name": args.name, "email": args.email,
               "phone": args.phone, "owner_id": None, "version": "v2"}
    CONTACTS[cid] = contact
    return contact


print(create_contact_v1(name="Old Caller", email="old@example.com"))
print(create_contact_v2(name="New Caller", email="new@example.com", phone="+1-555-0100"))

## Pattern 5 — Read vs write separation

A common mistake: every tool can mutate state. A safer design tags tools as `read` or `write`, and the auth layer only hands `write` tools to callers with the right scope.

In [ ]:
@logged("list_contacts")
def list_contacts() -> list[dict]:
    """Read-only: list all contacts."""
    return list(CONTACTS.values())


@logged("get_contact")
def get_contact(contact_id: str) -> dict:
    """Read-only: fetch one contact."""
    if contact_id not in CONTACTS:
        raise NotFoundError(f"contact {contact_id} not found")
    return CONTACTS[contact_id]


TOOL_REGISTRY = {
    "list_contacts":      {"fn": list_contacts,      "kind": "read"},
    "get_contact":        {"fn": get_contact,        "kind": "read"},
    "create_contact":     {"fn": create_contact_v1,  "kind": "write"},
    "create_contact_v2":  {"fn": create_contact_v2,  "kind": "write"},
}


def call_with_scope(name: str, args: dict, *, scope: str) -> dict:
    if name not in TOOL_REGISTRY:
        return ToolError("unknown_tool", name).to_response()
    entry = TOOL_REGISTRY[name]
    if entry["kind"] == "write" and scope != "write":
        return ToolError("forbidden",
                         f"tool {name} requires 'write' scope, caller has '{scope}'").to_response()
    return safe_call(entry["fn"], **args)

## Putting it together

In [ ]:
# Read-only caller can browse but not write
print(call_with_scope("list_contacts", {}, scope="read"))
print(call_with_scope("create_contact",
                      {"name": "Eve", "email": "eve@example.com"},
                      scope="read"))

# Write-scope caller can do everything
print(call_with_scope("create_contact",
                      {"name": "Frank", "email": "frank@example.com"},
                      scope="write"))

# Invalid input is caught even with write scope
print(call_with_scope("create_contact",
                      {"name": "", "email": "nope"},
                      scope="write"))

## Mini test

In [ ]:
ok = call_with_scope("create_contact",
                     {"name": "Grace", "email": "grace@example.com"},
                     scope="write")
assert ok["ok"] is True
assert ok["result"]["version"] == "v1"

denied = call_with_scope("create_contact",
                         {"name": "Heidi", "email": "h@example.com"},
                         scope="read")
assert denied["error"]["code"] == "forbidden"

invalid = call_with_scope("create_contact", {"name": "", "email": "x"}, scope="write")
assert invalid["error"]["code"] == "validation_failed"

print("ok")

## Key takeaway

The shape of "production MCP" is the same as any other production service: **validate everything, log everything, version everything, and split read from write**. None of these are MCP-specific — but every one of them matters more when an LLM is making the calls, because the LLM will absolutely produce surprising arguments.